# LeetCode 647 — Palindromic Substrings bằng Python

Link đề: https://leetcode.com/problems/palindromic-substrings/ · Độ khó: **Medium** · Chủ đề: Two Pointers, String, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def countSubstrings(self, s: str) -> int:
        ...
```

Ghi chú: bài đếm **số chuỗi con (substring) là palindrome (chuỗi đối xứng)**, tính theo **vị trí** (hai chuỗi con giống nội dung nhưng ở vị trí khác nhau vẫn đếm riêng). Notebook đi từ vét cạn `O(n^3)`, qua DP `O(n^2)`, mở rộng quanh tâm `O(n^2)` với `O(1)` bộ nhớ, và thuật toán Manacher `O(n)`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm mọi chuỗi con **liên tiếp** của `s` đọc xuôi và ngược giống nhau, **đếm theo vị trí** (ví dụ `"aaa"` cho `6`, không phải `3`).
2. Nắm các công cụ Python: cắt chuỗi `s[i:j+1]`, đảo chuỗi `s[::-1]`, hai con trỏ `i, j`, bảng 2 chiều `[[False] * n for _ in range(n)]`, `'#'.join(...)` để chèn ký tự phân cách, `range` đảo chiều.
3. Đi từ **vét cạn mọi chuỗi con** → **DP `dp[i][j]`** → **mở rộng quanh tâm** → **Manacher**.
4. Hiểu mấu chốt: **một palindrome được xác định bởi tâm và bán kính**; mọi palindrome dài hơn đều chứa một palindrome ngắn hơn cùng tâm bên trong. Có đúng `2n - 1` tâm (`n` tâm ký tự + `n - 1` tâm "giữa hai ký tự").
5. Tránh các lỗi kinh điển: quên tâm chẵn, kiểm tra `s[i]` khi `i = -1` (Python **không báo lỗi** mà lấy ký tự cuối), thứ tự duyệt sai trong DP, đếm chuỗi con khác nhau thay vì theo vị trí, và dùng `O(n^3)` cho `n = 1000`.

## 2. Đề bài & ví dụ

Cho chuỗi `s`, hãy trả về **số chuỗi con palindrome** của nó.

Palindrome là chuỗi đọc xuôi và đọc ngược giống nhau. Chuỗi con (substring) là dãy ký tự **liên tiếp** trong `s`.

**Ví dụ 1**

- Input: `s = "abc"`
- Output: `3`
- Giải thích: ba chuỗi `"a"`, `"b"`, `"c"`.

**Ví dụ 2**

- Input: `s = "aaa"`
- Output: `6`
- Giải thích: `"a"`, `"a"`, `"a"`, `"aa"`, `"aa"`, `"aaa"` — các chuỗi trùng nội dung nhưng khác vị trí vẫn đếm riêng.

**Ràng buộc:**

- `1 <= s.length <= 1000`
- `s` chỉ gồm chữ cái tiếng Anh viết thường.

Hệ quả: `n` tối đa `1000` nên `O(n^2)` (khoảng `10^6`) rất thoải mái, còn `O(n^3)` (khoảng `10^9`) thì quá chậm.

In [1]:
from typing import List
import random

examples = [
    ("abc", 3),
    ("aaa", 6),
]
for s, expected in examples:
    print(repr(s), "n =", len(s), "-> kỳ vọng", expected)

'abc' n = 3 -> kỳ vọng 3
'aaa' n = 3 -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1. Palindrome và cắt chuỗi

Chuỗi trong Python là dãy ký tự đánh số từ `0`. Cắt chuỗi `s[i:j]` lấy các ký tự từ chỉ số `i` tới `j - 1` (**không** gồm `j`). Vì vậy chuỗi con từ chỉ số `i` tới `j` **bao gồm** cả hai đầu là `s[i:j+1]`. Cắt `s[::-1]` cho chuỗi **đảo ngược**. Một chuỗi là palindrome khi `t == t[::-1]`.

In [2]:
s = "abcba"
print(s[1:4])          # 'bcb'  (chỉ số 1, 2, 3)
print(s[0:0 + 1])      # 'a'    (chuỗi con đúng 1 ký tự, chỉ số 0)
print(s[::-1])         # 'abcba'
for t in ["aba", "abba", "abc", "a", ""]:
    print(repr(t), "palindrome?", t == t[::-1])

bcb
a
abcba
'aba' palindrome? True
'abba' palindrome? True
'abc' palindrome? False
'a' palindrome? True
'' palindrome? True


### 3.2. Có bao nhiêu chuỗi con?

Chuỗi con được xác định bởi cặp `(i, j)` với `0 <= i <= j <= n - 1`. Có `n` cặp có `i = j`, `n - 1` cặp có `j = i + 1`, ..., tổng cộng `n + (n-1) + ... + 1 = n(n+1)/2`. Với `n = 1000` là `500500` chuỗi con — đủ nhỏ để liệt kê, nhưng mỗi lần kiểm tra palindrome tốn `O(độ dài)` nên tổng `O(n^3)`.

In [3]:
s = "abc"
subs = [(i, j, s[i:j + 1]) for i in range(len(s)) for j in range(i, len(s))]
for i, j, t in subs:
    print(f"({i},{j}) {t!r}")
n = len(s)
print("số chuỗi con:", len(subs), "= n(n+1)/2 =", n * (n + 1) // 2)

(0,0) 'a'
(0,1) 'ab'
(0,2) 'abc'
(1,1) 'b'
(1,2) 'bc'
(2,2) 'c'
số chuỗi con: 6 = n(n+1)/2 = 6


### 3.3. Hai con trỏ `i`, `j` và vòng `while`

Ta dùng hai chỉ số `i` (đi sang trái) và `j` (đi sang phải), kiểm tra `s[i] == s[j]`, rồi mở rộng `i -= 1`, `j += 1`. Điều kiện dừng quan trọng: **`i >= 0` và `j < n`**. Cẩn thận: chỉ số **âm** trong Python là hợp lệ (`s[-1]` là ký tự cuối), nên nếu quên kiểm tra `i >= 0`, chương trình sẽ **âm thầm** so sánh sai thay vì báo lỗi.

In [4]:
s = "abc"
print(s[-1], s[-2])              # 'c' 'b' : chỉ số âm đếm từ cuối!
i = -1
print("s[i] với i = -1 ->", s[i], "(không lỗi, nhưng không phải 'ký tự bên trái ký tự đầu')")

def is_palindrome_two_pointers(t):
    i, j = 0, len(t) - 1
    while i < j:
        if t[i] != t[j]:
            return False
        i, j = i + 1, j - 1
    return True

print([is_palindrome_two_pointers(t) for t in ["aba", "abba", "abc", "a"]])

c b
s[i] với i = -1 -> c (không lỗi, nhưng không phải 'ký tự bên trái ký tự đầu')
[True, True, False, True]


### 3.4. Bảng 2 chiều và vòng lặp đảo chiều

Với DP ta cần bảng `dp[i][j]`. Tạo bằng `[[False] * n for _ in range(n)]` (không dùng `[[False] * n] * n` vì các hàng sẽ là cùng một list). Duyệt `i` từ cuối về đầu bằng `range(n - 1, -1, -1)`: bắt đầu `n - 1`, dừng trước `-1`, bước `-1`.

In [5]:
n = 4
dp = [[False] * n for _ in range(n)]
dp[0][1] = True
print(dp)
print(list(range(n - 1, -1, -1)))     # [3, 2, 1, 0]

[[False, True, False, False], [False, False, False, False], [False, False, False, False], [False, False, False, False]]
[3, 2, 1, 0]


### 3.5. `'#'.join(...)` — chèn ký tự phân cách (dùng cho Manacher)

`'#'.join("aba")` nối các ký tự bằng `#`: `"a#b#a"`. Thêm `#` ở hai đầu: `"#a#b#a#"`. Mọi palindrome (chẵn hay lẻ) trong chuỗi gốc trở thành palindrome **độ dài lẻ** có tâm ở một ký tự của chuỗi mới.

In [6]:
s = "abba"
t = "#" + "#".join(s) + "#"
print(t, "| độ dài", len(t), "= 2n + 1 =", 2 * len(s) + 1)

#a#b#b#a# | độ dài 9 = 2n + 1 = 9


## 4. Ý tưởng nền tảng / trực giác

Hãy tự liệt kê với `s = "aaa"`. Tất cả `6` chuỗi con (theo vị trí):

| `(i, j)` | chuỗi con | palindrome? |
|---|---|---|
| (0,0) | a | có |
| (1,1) | a | có |
| (2,2) | a | có |
| (0,1) | aa | có |
| (1,2) | aa | có |
| (0,2) | aaa | có |

Còn với `"abc"`: `(0,1) = "ab"`, `(1,2) = "bc"`, `(0,2) = "abc"` đều **không** phải palindrome, nên chỉ `3` chuỗi 1 ký tự.

**Quan sát then chốt (đối xứng):** nếu `s[i..j]` là palindrome thì `s[i+1..j-1]` (bỏ hai đầu) cũng là palindrome **và** `s[i] == s[j]`. Ngược lại, nếu `s[i+1..j-1]` là palindrome và `s[i] == s[j]` thì `s[i..j]` là palindrome. Nghĩa là palindrome **mọc dần từ giữa ra hai bên** — từ đây có hai hướng: ghi nhớ bảng `dp[i][j]`, hoặc chọn tâm rồi mở rộng.

In [7]:
def list_palindromes(s):
    out = []
    for i in range(len(s)):
        for j in range(i, len(s)):
            t = s[i:j + 1]
            if t == t[::-1]:
                out.append((i, j, t))
    return out

for s in ["aaa", "abc", "abba"]:
    ps = list_palindromes(s)
    print(repr(s), "->", len(ps), ps)

'aaa' -> 6 [(0, 0, 'a'), (0, 1, 'aa'), (0, 2, 'aaa'), (1, 1, 'a'), (1, 2, 'aa'), (2, 2, 'a')]
'abc' -> 3 [(0, 0, 'a'), (1, 1, 'b'), (2, 2, 'c')]
'abba' -> 6 [(0, 0, 'a'), (0, 3, 'abba'), (1, 1, 'b'), (1, 2, 'bb'), (2, 2, 'b'), (3, 3, 'a')]


## 5. Cách 1 — Brute Force: kiểm tra mọi chuỗi con

### Ý tưởng

Liệt kê mọi cặp `(i, j)`, cắt `t = s[i:j+1]`, so `t` với `t[::-1]`, đếm nếu bằng nhau.

### Code

In [8]:
def count_brute(s):
    n = len(s)
    ans = 0
    for i in range(n):
        for j in range(i, n):
            t = s[i:j + 1]
            if t == t[::-1]:
                ans += 1
    return ans

for s, expected in examples:
    print(repr(s), "->", count_brute(s), "(kỳ vọng", expected, ")")
    assert count_brute(s) == expected

'abc' -> 3 (kỳ vọng 3 )
'aaa' -> 6 (kỳ vọng 6 )


### Độ phức tạp và nút thắt

- Có `O(n^2)` chuỗi con, mỗi lần cắt + đảo + so sánh tốn `O(n)` → Time `O(n^3)`.
- Space `O(n)` cho chuỗi con tạm.

Với `n = 1000`: khoảng `n^3 / 6 ≈ 1.7 * 10^8` thao tác ký tự — rất chậm. **Nút thắt:** mỗi lần kiểm tra palindrome ta **bỏ qua** kết quả của chuỗi con bên trong đã biết (`s[i+1..j-1]`). Hãy đếm xem lãng phí cỡ nào.

In [9]:
def brute_char_ops(n):
    """Ước lượng tổng số ký tự phải so sánh (tối đa) của cách vét cạn với chuỗi dài n."""
    return sum(j - i + 1 for i in range(n) for j in range(i, n))

for n in [10, 50, 100, 200]:
    print(f"n={n}: ~{brute_char_ops(n)} ký tự cắt/đảo; n^3/6 = {n**3 // 6}")
print("n=1000 sẽ cỡ", 1000 ** 3 // 6)

n=10: ~220 ký tự cắt/đảo; n^3/6 = 166
n=50: ~22100 ký tự cắt/đảo; n^3/6 = 20833
n=100: ~171700 ký tự cắt/đảo; n^3/6 = 166666
n=200: ~1353400 ký tự cắt/đảo; n^3/6 = 1333333
n=1000 sẽ cỡ 166666666


## 6. Ý tưởng thuật toán / bước đột phá

**Palindrome = tâm + bán kính.** Nếu `s[i..j]` là palindrome thì khi thu nhỏ đối xứng (bỏ đều hai đầu) ta vẫn được palindrome **cùng tâm**. Vậy với mỗi tâm, các palindrome nhận tâm đó tạo thành một dãy **liên tục** từ nhỏ tới một bán kính tối đa `R`: nếu mở rộng thất bại ở bước `k` thì mọi bước sau cũng thất bại.

Hệ quả: với **mỗi tâm**, chỉ cần mở rộng từng bước ra hai bên, **mỗi bước thành công cộng 1** vào đáp án, dừng ngay ở bước thất bại đầu tiên. Không cần kiểm tra lại từ đầu.

Có bao nhiêu tâm? Palindrome độ dài **lẻ** có tâm ở một ký tự: `n` tâm. Palindrome độ dài **chẵn** có tâm ở **khoảng giữa** hai ký tự liền nhau: `n - 1` tâm. Tổng `2n - 1`.

Đánh số tâm bằng `k = 0 .. 2n - 2`: cho `k`, hai con trỏ ban đầu là `i = k // 2`, `j = (k + 1) // 2`. Khi `k` chẵn thì `i = j` (tâm lẻ, cùng một ký tự); khi `k` lẻ thì `j = i + 1` (tâm chẵn, giữa hai ký tự).

In [10]:
n = 4
for k in range(2 * n - 1):
    i, j = k // 2, (k + 1) // 2
    kind = "lẻ (một ký tự)" if i == j else "chẵn (giữa hai ký tự)"
    print(f"k={k}: i={i}, j={j} -> tâm {kind}")

k=0: i=0, j=0 -> tâm lẻ (một ký tự)
k=1: i=0, j=1 -> tâm chẵn (giữa hai ký tự)
k=2: i=1, j=1 -> tâm lẻ (một ký tự)
k=3: i=1, j=2 -> tâm chẵn (giữa hai ký tự)
k=4: i=2, j=2 -> tâm lẻ (một ký tự)
k=5: i=2, j=3 -> tâm chẵn (giữa hai ký tự)
k=6: i=3, j=3 -> tâm lẻ (một ký tự)


## 7. Các cách tiếp theo

Phần này có thêm 3 cách (tổng 4 cách): **DP**, **mở rộng quanh tâm** (lời giải nộp), **Manacher**.

### 7.1. Cách 2 — Quy hoạch động `dp[i][j]`

**Giải thích từng đoạn.** Gọi `dp[i][j]` = "chuỗi con `s[i..j]` có là palindrome không". Quy tắc:

- `i == j`: một ký tự → `True`.
- `j == i + 1`: hai ký tự → `True` nếu `s[i] == s[j]`.
- `j >= i + 2`: `True` khi `s[i] == s[j]` **và** `dp[i+1][j-1]` là `True`.

(Gộp: `dp[i][j] = s[i] == s[j] and (j - i < 3 or dp[i+1][j-1])`.)

**Thứ tự điền:** `dp[i][j]` phụ thuộc `dp[i+1][j-1]` (hàng `i + 1`, tức hàng **sau**). Vì vậy duyệt `i` **từ `n - 1` về `0`**, với mỗi `i` duyệt `j` từ `i` tới `n - 1`. Mỗi ô `True` cộng 1 vào đáp án.

In [11]:
def count_dp(s):
    n = len(s)
    dp = [[False] * n for _ in range(n)]
    ans = 0
    for i in range(n - 1, -1, -1):          # i giảm dần: hàng i + 1 đã tính xong
        for j in range(i, n):
            if s[i] == s[j] and (j - i < 3 or dp[i + 1][j - 1]):
                dp[i][j] = True
                ans += 1
    return ans

for s, expected in examples:
    print(repr(s), "->", count_dp(s))
    assert count_dp(s) == expected

'abc' -> 3
'aaa' -> 6


In [12]:
class Solution:
    def countSubstrings(self, s: str) -> int:
        n = len(s)
        dp = [[False] * n for _ in range(n)]
        ans = 0
        for i in range(n - 1, -1, -1):
            for j in range(i, n):
                if s[i] == s[j] and (j - i < 3 or dp[i + 1][j - 1]):
                    dp[i][j] = True
                    ans += 1
        return ans

print(Solution().countSubstrings("aaa"))

6


**Độ phức tạp:** Time `O(n^2)` (mỗi ô `O(1)`), Space `O(n^2)` cho bảng. Với `n = 1000` là `10^6` ô — chạy được nhưng tốn bộ nhớ so với cách sau.

### 7.2. Cách 3 — Mở rộng quanh tâm (lời giải nộp)

**Giải thích từng đoạn.**

1. Duyệt `k` từ `0` tới `2n - 2`; `i = k // 2`, `j = (k + 1) // 2`.
2. `while i >= 0 and j < n and s[i] == s[j]`: ta vừa tìm được một palindrome `s[i..j]` → `ans += 1`; rồi mở rộng `i -= 1`, `j += 1`.
3. Ra khỏi vòng `while` thì chuyển sang tâm tiếp theo.

Thứ tự điều kiện quan trọng: **kiểm tra biên trước, so ký tự sau** (`and` dừng ngay ở điều kiện sai đầu tiên, nên không bao giờ đọc `s[-1]` hay `s[n]`).

In [13]:
def count_expand(s):
    n = len(s)
    ans = 0
    for k in range(2 * n - 1):
        i, j = k // 2, (k + 1) // 2
        while i >= 0 and j < n and s[i] == s[j]:
            ans += 1
            i, j = i - 1, j + 1
    return ans

for s, expected in examples:
    print(repr(s), "->", count_expand(s))
    assert count_expand(s) == expected

'abc' -> 3
'aaa' -> 6


In [14]:
class Solution:
    def countSubstrings(self, s: str) -> int:
        n = len(s)
        ans = 0
        for k in range(2 * n - 1):
            i, j = k // 2, (k + 1) // 2
            while i >= 0 and j < n and s[i] == s[j]:
                ans += 1
                i, j = i - 1, j + 1
        return ans

print(Solution().countSubstrings("abc"), Solution().countSubstrings("aaa"))

3 6


**Độ phức tạp:** Time `O(n^2)` (tệ nhất với `"aaaa..."`: mỗi tâm mở rộng tới `O(n)`), Space `O(1)`. Đơn giản, nhanh trong thực tế (đa số tâm dừng rất sớm) và không tốn bộ nhớ.

### 7.3. Cách 4 — Thuật toán Manacher `O(n)`

**Ý tưởng.** Cách 3 mở rộng lại từ đầu cho mỗi tâm, dù các tâm gần nhau chia sẻ nhiều thông tin. Manacher tận dụng **tính đối xứng của palindrome lớn**: nếu ta đã biết một palindrome lớn tâm `c`, phủ tới biên phải `r`, thì với tâm `i` nằm trong `(c, r]`, bán kính của `i` **ít nhất** là `min(r - i, p[2c - i])` (`2c - i` là điểm đối xứng với `i` qua `c`). Chỉ cần mở rộng thêm từ đó.

**Bước chuẩn bị:** chèn `#` để xử lý chung tâm chẵn/lẻ: `t = "#" + "#".join(s) + "#"`, độ dài `2n + 1`. `p[i]` là bán kính lớn nhất sao cho `t[i-p..i+p]` là palindrome; bán kính này **đúng bằng độ dài** palindrome gốc có tâm tại vị trí đó.

**Đếm:** mỗi tâm `i` đóng góp `(p[i] + 1) // 2` palindrome trong chuỗi gốc (tâm ký tự có độ dài `1, 3, 5, ...`; tâm `#` có độ dài `2, 4, ...`).

Ví dụ `"aaa"`: `t = #a#a#a#`, `p = [0,1,2,3,2,1,0]`, đóng góp `[0,1,1,2,1,1,0]`, tổng `6`.

In [15]:
def count_manacher(s):
    t = "#" + "#".join(s) + "#"
    n = len(t)
    p = [0] * n
    c = r = 0                                   # c: tâm palindrome phủ xa nhất, r: biên phải của nó
    for i in range(n):
        if i < r:
            p[i] = min(r - i, p[2 * c - i])     # tận dụng điểm đối xứng
        a, b = i - p[i] - 1, i + p[i] + 1
        while a >= 0 and b < n and t[a] == t[b]:
            p[i] += 1
            a, b = a - 1, b + 1
        if i + p[i] > r:
            c, r = i, i + p[i]
    return sum((x + 1) // 2 for x in p)

for s, expected in examples:
    print(repr(s), "->", count_manacher(s))
    assert count_manacher(s) == expected

'abc' -> 3
'aaa' -> 6


In [16]:
class Solution:
    def countSubstrings(self, s: str) -> int:
        t = "#" + "#".join(s) + "#"
        n = len(t)
        p = [0] * n
        c = r = 0
        for i in range(n):
            if i < r:
                p[i] = min(r - i, p[2 * c - i])
            a, b = i - p[i] - 1, i + p[i] + 1
            while a >= 0 and b < n and t[a] == t[b]:
                p[i] += 1
                a, b = a - 1, b + 1
            if i + p[i] > r:
                c, r = i, i + p[i]
        return sum((x + 1) // 2 for x in p)

print(Solution().countSubstrings("aaa"))

6


**Độ phức tạp:** Time `O(n)` (biên phải `r` chỉ tăng, mỗi lần mở rộng thành công đẩy `r` lên nên tổng bước mở rộng `O(n)`), Space `O(n)` cho `t` và `p`. Nhanh nhất về lý thuyết nhưng khó nghĩ và dễ sai; với `n <= 1000` không cần thiết.

## 8. TRACE từng bước

### 8.1. Trace mở rộng quanh tâm với `s = "aaa"`

| `k` | `i` | `j` | điều kiện / so sánh | `ans` sau |
|---|---|---|---|---|
| 0 | 0 | 0 | `a == a` → +1 | 1 |
| 0 | -1 | 1 | `i < 0` dừng | 1 |
| 1 | 0 | 1 | `a == a` (`"aa"`) → +1 | 2 |
| 1 | -1 | 2 | dừng | 2 |
| 2 | 1 | 1 | `a == a` (`"a"`) → +1 | 3 |
| 2 | 0 | 2 | `a == a` (`"aaa"`) → +1 | 4 |
| 2 | -1 | 3 | dừng | 4 |
| 3 | 1 | 2 | `a == a` (`"aa"`) → +1 | 5 |
| 3 | 0 | 3 | `j = 3 = n` dừng | 5 |
| 4 | 2 | 2 | `a == a` → +1 | 6 |
| 4 | 1 | 3 | `j = n` dừng | 6 |

In [17]:
def trace_expand(s):
    n = len(s)
    ans = 0
    print(f"s = {s!r}, n = {n}, số tâm = {2 * n - 1}")
    for k in range(2 * n - 1):
        i, j = k // 2, (k + 1) // 2
        print(f"tâm k={k}: bắt đầu i={i}, j={j}")
        while i >= 0 and j < n and s[i] == s[j]:
            ans += 1
            print(f"    palindrome {s[i:j + 1]!r} (i={i}, j={j}) -> ans = {ans}")
            i, j = i - 1, j + 1
        why = "ra ngoài biên" if (i < 0 or j >= n) else f"{s[i]!r} != {s[j]!r}"
        print(f"    dừng ở i={i}, j={j}: {why}")
    return ans

assert trace_expand("aaa") == 6

s = 'aaa', n = 3, số tâm = 5
tâm k=0: bắt đầu i=0, j=0
    palindrome 'a' (i=0, j=0) -> ans = 1
    dừng ở i=-1, j=1: ra ngoài biên
tâm k=1: bắt đầu i=0, j=1
    palindrome 'aa' (i=0, j=1) -> ans = 2
    dừng ở i=-1, j=2: ra ngoài biên
tâm k=2: bắt đầu i=1, j=1
    palindrome 'a' (i=1, j=1) -> ans = 3
    palindrome 'aaa' (i=0, j=2) -> ans = 4
    dừng ở i=-1, j=3: ra ngoài biên
tâm k=3: bắt đầu i=1, j=2
    palindrome 'aa' (i=1, j=2) -> ans = 5
    dừng ở i=0, j=3: ra ngoài biên
tâm k=4: bắt đầu i=2, j=2
    palindrome 'a' (i=2, j=2) -> ans = 6
    dừng ở i=1, j=3: ra ngoài biên


In [18]:
assert trace_expand("abba") == 6     # có cả tâm chẵn: "bb" và "abba"

s = 'abba', n = 4, số tâm = 7
tâm k=0: bắt đầu i=0, j=0
    palindrome 'a' (i=0, j=0) -> ans = 1
    dừng ở i=-1, j=1: ra ngoài biên
tâm k=1: bắt đầu i=0, j=1
    dừng ở i=0, j=1: 'a' != 'b'
tâm k=2: bắt đầu i=1, j=1
    palindrome 'b' (i=1, j=1) -> ans = 2
    dừng ở i=0, j=2: 'a' != 'b'
tâm k=3: bắt đầu i=1, j=2
    palindrome 'bb' (i=1, j=2) -> ans = 3
    palindrome 'abba' (i=0, j=3) -> ans = 4
    dừng ở i=-1, j=4: ra ngoài biên
tâm k=4: bắt đầu i=2, j=2
    palindrome 'b' (i=2, j=2) -> ans = 5
    dừng ở i=1, j=3: 'b' != 'a'
tâm k=5: bắt đầu i=2, j=3
    dừng ở i=2, j=3: 'b' != 'a'
tâm k=6: bắt đầu i=3, j=3
    palindrome 'a' (i=3, j=3) -> ans = 6
    dừng ở i=2, j=4: ra ngoài biên


### 8.2. Trace DP với `s = "aba"`

Duyệt `i = 2, 1, 0`. Ô `(i, j)` chỉ xét khi `j >= i`:

| `(i, j)` | `s[i] == s[j]`? | `j - i < 3` hoặc `dp[i+1][j-1]` | `dp[i][j]` |
|---|---|---|---|
| (2,2) | a==a | `j-i=0 < 3` | True |
| (1,1) | b==b | `0 < 3` | True |
| (1,2) | b!=a | | False |
| (0,0) | a==a | `0 < 3` | True |
| (0,1) | a!=b | | False |
| (0,2) | a==a | `j-i = 2 < 3` | True (`"aba"`) |

Đáp án `4`.

In [19]:
def trace_dp(s):
    n = len(s)
    dp = [[False] * n for _ in range(n)]
    ans = 0
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            eq = s[i] == s[j]
            inner = (j - i < 3) or dp[i + 1][j - 1]
            dp[i][j] = eq and inner
            ans += dp[i][j]
            print(f"({i},{j}) {s[i:j + 1]!r:6} s[i]==s[j]: {eq!s:5} inner ok: {inner!s:5} -> dp = {dp[i][j]}")
    print("bảng dp (T = True):")
    for row in dp:
        print(" ".join("T" if x else "." for x in row))
    return ans

assert trace_dp("aba") == 4

(2,2) 'a'    s[i]==s[j]: True  inner ok: True  -> dp = True
(1,1) 'b'    s[i]==s[j]: True  inner ok: True  -> dp = True
(1,2) 'ba'   s[i]==s[j]: False inner ok: True  -> dp = False
(0,0) 'a'    s[i]==s[j]: True  inner ok: True  -> dp = True
(0,1) 'ab'   s[i]==s[j]: False inner ok: True  -> dp = False
(0,2) 'aba'  s[i]==s[j]: True  inner ok: True  -> dp = True
bảng dp (T = True):
T . T
. T .
. . T


### 8.3. Trace Manacher với `s = "aaa"`

`t = "#a#a#a#"`, `n = 7`. Bảng: `i`, `c`/`r` trước khi xử lý, bán kính khởi đầu (từ đối xứng), bán kính cuối.

| `i` | `c, r` trước | khởi đầu `p[i]` | mở rộng | `p[i]` cuối | `c, r` sau |
|---|---|---|---|---|---|
| 0 | 0, 0 | 0 | không được | 0 | 0, 0 |
| 1 | 0, 0 | 0 | +1 | 1 | 1, 2 |
| 2 | 1, 2 | 0 (`i < r` sai) | +2 | 2 | 2, 4 |
| 3 | 2, 4 | `min(4-3, p[1]=1) = 1` | +2 | 3 | 3, 6 |
| 4 | 3, 6 | `min(6-4, p[2]=2) = 2` | không mở rộng thêm | 2 | 3, 6 |
| 5 | 3, 6 | `min(6-5, p[1]=1) = 1` | không | 1 | 3, 6 |
| 6 | 3, 6 | `i < r` sai → 0 | không | 0 | 3, 6 |

`p = [0,1,2,3,2,1,0]`. Ở `i = 4, 5` ta **không phải so sánh gì**, bán kính suy ra từ đối xứng: đó là chỗ Manacher tiết kiệm.

In [20]:
def trace_manacher(s):
    t = "#" + "#".join(s) + "#"
    n = len(t)
    p = [0] * n
    c = r = 0
    print("t =", t)
    for i in range(n):
        before = (c, r)
        if i < r:
            p[i] = min(r - i, p[2 * c - i])
        start = p[i]
        a, b = i - p[i] - 1, i + p[i] + 1
        grow = 0
        while a >= 0 and b < n and t[a] == t[b]:
            p[i] += 1
            grow += 1
            a, b = a - 1, b + 1
        if i + p[i] > r:
            c, r = i, i + p[i]
        print(f"i={i} ({t[i]!r}): (c,r) trước={before}, p khởi đầu={start}, mở rộng +{grow}, p[i]={p[i]}, (c,r) sau={(c, r)}")
    contrib = [(x + 1) // 2 for x in p]
    print("p =", p)
    print("đóng góp =", contrib, "-> tổng", sum(contrib))
    return sum(contrib)

assert trace_manacher("aaa") == 6

t = #a#a#a#
i=0 ('#'): (c,r) trước=(0, 0), p khởi đầu=0, mở rộng +0, p[i]=0, (c,r) sau=(0, 0)
i=1 ('a'): (c,r) trước=(0, 0), p khởi đầu=0, mở rộng +1, p[i]=1, (c,r) sau=(1, 2)
i=2 ('#'): (c,r) trước=(1, 2), p khởi đầu=0, mở rộng +2, p[i]=2, (c,r) sau=(2, 4)
i=3 ('a'): (c,r) trước=(2, 4), p khởi đầu=1, mở rộng +2, p[i]=3, (c,r) sau=(3, 6)
i=4 ('#'): (c,r) trước=(3, 6), p khởi đầu=2, mở rộng +0, p[i]=2, (c,r) sau=(3, 6)
i=5 ('a'): (c,r) trước=(3, 6), p khởi đầu=1, mở rộng +0, p[i]=1, (c,r) sau=(3, 6)
i=6 ('#'): (c,r) trước=(3, 6), p khởi đầu=0, mở rộng +0, p[i]=0, (c,r) sau=(3, 6)
p = [0, 1, 2, 3, 2, 1, 0]
đóng góp = [0, 1, 1, 2, 1, 1, 0] -> tổng 6


## 9. Code nộp LeetCode

Lời giải nộp là **mở rộng quanh tâm** (Cách 3): ngắn, `O(1)` bộ nhớ, `O(n^2)` thời gian là dư sức với `n <= 1000`.

In [21]:
class Solution:
    def countSubstrings(self, s: str) -> int:
        n = len(s)
        ans = 0
        for k in range(2 * n - 1):
            i, j = k // 2, (k + 1) // 2
            while i >= 0 and j < n and s[i] == s[j]:
                ans += 1
                i, j = i - 1, j + 1
        return ans


sol = Solution()
for s, expected in examples:
    got = sol.countSubstrings(s)
    print(repr(s), "->", got)
    assert got == expected

'abc' -> 3
'aaa' -> 6


## 10. Vì sao đúng

**Mệnh đề 1 (song ánh palindrome ↔ (tâm, bước)).** Mỗi chuỗi con `s[i..j]` có đúng một tâm: `k = i + j` (số nguyên từ `0` đến `2n - 2`; ứng với `i' = k // 2`, `j' = (k + 1) // 2` là hai chỉ số ở giữa). Chuỗi con `s[i..j]` có tâm `k` đúng bằng chuỗi nhận được sau `t` bước mở rộng từ `(i', j')` với `t = i' - i`. Vậy mỗi chuỗi con tương ứng đúng một cặp `(tâm k, bước t)`, và ngược lại.

**Mệnh đề 2 (tính đơn điệu theo bước).** `s[i'-t .. j'+t]` là palindrome khi và chỉ khi `s[i'-t] == s[j'+t]` **và** `s[i'-t+1 .. j'+t-1]` là palindrome (bỏ hai đầu). Do đó nếu bước `t` thất bại (không palindrome) thì mọi bước `t' > t` cũng thất bại — vì chúng chứa chuỗi con bên trong không đối xứng hoặc có cặp đầu cuối khác nhau. Vòng `while` dừng ở bước thất bại đầu tiên, không bỏ sót palindrome nào.

**Kết luận.** Vòng `while` cộng 1 đúng một lần cho mỗi cặp `(tâm, bước)` mà chuỗi con tương ứng là palindrome — mỗi palindrome được đếm **đúng một lần**. Tổng chính là đáp án. Mọi truy cập `s[i]`, `s[j]` hợp lệ vì điều kiện biên đứng trước phép so sánh.

In [22]:
# Kiểm chứng thực nghiệm: mỗi cặp (i, j) có tâm k = i + j; số palindrome theo tâm cộng lại = tổng
from collections import Counter

def palindromes_by_center(s):
    c = Counter()
    for i, j, t in list_palindromes(s):
        c[i + j] += 1
    return c

def per_center_expand(s):
    n = len(s)
    out = {}
    for k in range(2 * n - 1):
        i, j = k // 2, (k + 1) // 2
        cnt = 0
        while i >= 0 and j < n and s[i] == s[j]:
            cnt += 1
            i, j = i - 1, j + 1
        if cnt:
            out[k] = cnt
    return out

rng = random.Random(10)
for _ in range(200):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 12)))
    # tâm k trong expand tương ứng i + j = 2 * (k // 2) + (k % 2) = k
    assert dict(palindromes_by_center(s)) == per_center_expand(s), s
print("mỗi palindrome gắn với đúng một tâm k = i + j, khớp với vòng mở rộng")

mỗi palindrome gắn với đúng một tâm k = i + j, khớp với vòng mở rộng


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n^3)` | `O(n)` |
| 2. DP `dp[i][j]` | `O(n^2)` | `O(n^2)` |
| 3. Mở rộng quanh tâm (nộp) | `O(n^2)` | `O(1)` |
| 4. Manacher | `O(n)` | `O(n)` |

Tệ nhất của cách 3 là chuỗi toàn một ký tự: mỗi tâm mở rộng tới sát biên. Ta đếm số bước mở rộng thật để kiểm chứng `Θ(n^2)`, và so sánh với chuỗi ngẫu nhiên (ít bước).

In [23]:
def expand_steps(s):
    n = len(s)
    steps = 0
    for k in range(2 * n - 1):
        i, j = k // 2, (k + 1) // 2
        while i >= 0 and j < n and s[i] == s[j]:
            steps += 1
            i, j = i - 1, j + 1
    return steps

rng = random.Random(11)
for n in [10, 100, 1000]:
    worst = "a" * n
    rand = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(n))
    print(f"n={n}: toàn 'a' -> {expand_steps(worst)} bước (n(n+1)/2 = {n * (n + 1) // 2}); ngẫu nhiên 26 chữ -> {expand_steps(rand)} bước")

n=10: toàn 'a' -> 55 bước (n(n+1)/2 = 55); ngẫu nhiên 26 chữ -> 11 bước
n=100: toàn 'a' -> 5050 bước (n(n+1)/2 = 5050); ngẫu nhiên 26 chữ -> 106 bước
n=1000: toàn 'a' -> 500500 bước (n(n+1)/2 = 500500); ngẫu nhiên 26 chữ -> 1082 bước


## 12. Bộ test

Gồm: ví dụ đề, biên (`n = 1`, toàn một ký tự, toàn khác nhau, xen kẽ), phản ví dụ cho các lời giải sai, đối chiếu 4 cách bằng `assert` trên nhiều input (kể cả **vét cạn mọi chuỗi** độ dài nhỏ trên bảng chữ cái `{a, b}`), test ngẫu nhiên có seed đúng ràng buộc, và test `n = 1000`.

In [24]:
ALL_METHODS = {
    "brute": count_brute,
    "dp": count_dp,
    "expand": count_expand,
    "manacher": count_manacher,
    "leetcode": Solution().countSubstrings,
}

fixed_cases = [
    ("abc", 3),
    ("aaa", 6),
    ("a", 1),
    ("aa", 3),
    ("ab", 2),
    ("abba", 6),
    ("aba", 4),
    ("abcba", 7),                 # a b c b a + bcb + abcba
    ("aaaa", 10),                 # n(n+1)/2
    ("abab", 6),                  # a b a b + aba + bab
    ("abcdefg", 7),               # toàn khác nhau
    ("aabaa", 9),
]
for s, expected in fixed_cases:
    for name, fn in ALL_METHODS.items():
        assert fn(s) == expected, (name, s, fn(s), expected)
print("test cố định:", len(fixed_cases), "bộ x", len(ALL_METHODS), "cách: OK")

test cố định: 12 bộ x 5 cách: OK


In [25]:
# Phản ví dụ cho lời giải thiếu tâm chẵn: chỉ xét tâm ký tự
def wrong_odd_only(s):
    n = len(s)
    ans = 0
    for i in range(n):
        l, r = i, i
        while l >= 0 and r < n and s[l] == s[r]:
            ans += 1
            l, r = l - 1, r + 1
    return ans

for s in ["aa", "abba", "aaa"]:
    print(f"{s!r}: thiếu tâm chẵn -> {wrong_odd_only(s)}, đúng -> {count_expand(s)}")

# Phản ví dụ: đếm chuỗi con KHÁC NHAU thay vì theo vị trí
def wrong_distinct(s):
    return len({t for _, _, t in list_palindromes(s)})

for s in ["aaa", "abab"]:
    print(f"{s!r}: đếm khác nhau -> {wrong_distinct(s)}, đúng (theo vị trí) -> {count_expand(s)}")

'aa': thiếu tâm chẵn -> 2, đúng -> 3
'abba': thiếu tâm chẵn -> 4, đúng -> 6
'aaa': thiếu tâm chẵn -> 4, đúng -> 6
'aaa': đếm khác nhau -> 3, đúng (theo vị trí) -> 6
'abab': đếm khác nhau -> 4, đúng (theo vị trí) -> 6


In [26]:
# Vét cạn: MỌI chuỗi trên bảng chữ cái {a, b} độ dài 1..12  (8190 chuỗi), mọi cách phải khớp
from itertools import product

total = 0
for n in range(1, 13):
    for tup in product("ab", repeat=n):
        s = "".join(tup)
        ref = count_brute(s)
        assert count_dp(s) == ref, s
        assert count_expand(s) == ref, s
        assert count_manacher(s) == ref, s
        total += 1
print("vét cạn", total, "chuỗi nhị phân: 4 cách đều khớp")

for n in range(1, 8):
    for tup in product("abc", repeat=n):
        s = "".join(tup)
        assert count_expand(s) == count_brute(s) == count_manacher(s), s
print("vét cạn mọi chuỗi {a,b,c} độ dài <= 7: OK")

vét cạn 8190 chuỗi nhị phân: 4 cách đều khớp
vét cạn mọi chuỗi {a,b,c} độ dài <= 7: OK


In [27]:
# Ngẫu nhiên có seed, đúng ràng buộc (chỉ chữ thường, n <= 1000)
rng = random.Random(647)
alphabets = ["a", "ab", "abc", "abcdefghijklmnopqrstuvwxyz"]

for _ in range(300):
    n = rng.randint(1, 40)
    s = "".join(rng.choice(rng.choice(alphabets)) for _ in range(n))
    ref = count_brute(s)
    for name, fn in ALL_METHODS.items():
        assert fn(s) == ref, (name, s)
print("ngẫu nhiên nhỏ: OK")

# n lớn: bỏ vét cạn O(n^3)
for _ in range(10):
    n = rng.randint(300, 1000)
    s = "".join(rng.choice("ab") for _ in range(n))
    a, b, c = count_dp(s), count_expand(s), count_manacher(s)
    assert a == b == c == Solution().countSubstrings(s)
print("ngẫu nhiên lớn (đến n = 1000): OK")

# Biên lớn nhất
assert Solution().countSubstrings("a" * 1000) == 1000 * 1001 // 2
assert count_manacher("a" * 1000) == count_dp("a" * 1000) == 500500
assert Solution().countSubstrings("ab" * 500) == count_manacher("ab" * 500)
big_pal = "".join(chr(97 + (i * i) % 26) for i in range(500)); big_pal = big_pal + big_pal[::-1]    # palindrome dài 1000
assert Solution().countSubstrings(big_pal) == count_manacher(big_pal) == count_dp(big_pal)
print("n = 1000: OK")

ngẫu nhiên nhỏ: OK


ngẫu nhiên lớn (đến n = 1000): OK
n = 1000: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét tâm ký tự, quên tâm chẵn

Nếu chỉ mở rộng từ `(i, i)` thì không bao giờ tìm được `"aa"`, `"abba"`, ... Đã thấy ở mục 12: `"aa"` cho `2` thay vì `3`. Cách sửa: thêm tâm `(i, i + 1)` (hoặc dùng công thức `k // 2`, `(k + 1) // 2`).

### Lỗi 2 — So sánh `s[i]` trước khi kiểm tra `i >= 0`

Python cho phép chỉ số âm, nên `s[-1]` **không** gây lỗi. Nếu đặt `s[i] == s[j]` **trước** `i >= 0`, khi `i = -1` ta so với ký tự cuối chuỗi và có thể đếm thừa.

In [28]:
def wrong_check_order(s):
    n = len(s)
    ans = 0
    for k in range(2 * n - 1):
        i, j = k // 2, (k + 1) // 2
        while j < n and s[i] == s[j]:           # SAI: bỏ i >= 0, i = -1 sẽ lấy s[-1]
            ans += 1
            i, j = i - 1, j + 1
            if i < -1:                          # chỉ để chương trình không chạy mãi
                break
    return ans

for s in ["abb", "abca"]:
    print(f"{s!r}: sai -> {wrong_check_order(s)}, đúng -> {count_expand(s)}")

'abb': sai -> 5, đúng -> 4
'abca': sai -> 4, đúng -> 4


Với `"abb"`: đúng là `4` (`a`, `b`, `b`, `bb`). Nhưng khi mở rộng tâm `(0, 0)` ta có `i = -1`, `j = 1` và `s[-1] = 'b'` **trùng** `s[1] = 'b'`, nên đếm thừa một palindrome không tồn tại — kết quả sai mà **không có thông báo lỗi nào**. Luôn kiểm tra biên **trước**.

### Lỗi 3 — Sai thứ tự duyệt trong DP

`dp[i][j]` cần `dp[i+1][j-1]`. Nếu duyệt `i` tăng dần (`0, 1, 2, ...`) thì lúc tính hàng `i`, hàng `i + 1` chưa được điền (vẫn toàn `False`) → bỏ sót mọi palindrome dài từ 3 trở lên.

In [29]:
def wrong_dp_order(s):
    n = len(s)
    dp = [[False] * n for _ in range(n)]
    ans = 0
    for i in range(n):                          # SAI: i tăng dần
        for j in range(i, n):
            if s[i] == s[j] and (j - i < 3 or dp[i + 1][j - 1]):
                dp[i][j] = True
                ans += 1
    return ans

for s in ["aaa", "abcba", "aaaaa"]:
    print(f"{s!r}: sai -> {wrong_dp_order(s)}, đúng -> {count_dp(s)}")

'aaa': sai -> 6, đúng -> 6
'abcba': sai -> 6, đúng -> 7
'aaaaa': sai -> 12, đúng -> 15


### Lỗi 4 — Đếm chuỗi con khác nhau (dùng `set`)

Đề đếm theo **vị trí**: `"aaa"` có `6`, không phải `3`. Đã chứng minh ở mục 12 (`wrong_distinct`). Hãy đọc kỹ ví dụ 2 của đề: nó liệt kê `"a", "a", "a", ...` — các bản trùng nội dung vẫn tính riêng.

### Lỗi 5 — Dùng `O(n^3)` vét cạn cho `n = 1000`

Cách 1 đúng về kết quả nhưng quá chậm. Ta đo số ký tự phải xử lý để thấy khoảng cách với `O(n^2)`.

In [30]:
n = 1000
print("vét cạn ~ ký tự phải đụng tới:", n ** 3 // 6)
print("mở rộng tâm tối đa ~ bước:", n * (n + 1) // 2)
print("tỷ lệ ~", (n ** 3 // 6) // (n * (n + 1) // 2), "lần")

vét cạn ~ ký tự phải đụng tới: 166666666
mở rộng tâm tối đa ~ bước: 500500
tỷ lệ ~ 333 lần


## 14. Mẹo & nhận dạng pattern

- **"Đếm/tìm palindrome con"** → nghĩ ngay tới **mở rộng quanh tâm** (có `2n - 1` tâm) hoặc DP `dp[i][j]`.
- **Hai loại tâm** (lẻ/chẵn) là chỗ hay sót. Công thức gọn: `k` từ `0` đến `2n - 2`, `i = k // 2`, `j = (k + 1) // 2`.
- Khi chỉ cần **đếm** (không cần liệt kê chi tiết), mở rộng quanh tâm `O(1)` bộ nhớ thường tốt hơn DP `O(n^2)` bộ nhớ.
- Cần `O(n)` (độ dài `10^5`, `10^6`) → **Manacher**. Nhớ mẹo chèn `#` để gộp tâm chẵn/lẻ.
- Quy tắc vàng khi dùng chỉ số trong Python: **kiểm tra biên trước, truy cập sau**; nhớ chỉ số âm **không** báo lỗi.
- Với DP kiểu `dp[i][j]` phụ thuộc `dp[i+1][j-1]`: duyệt `i` giảm dần, hoặc duyệt theo **độ dài** chuỗi con tăng dần.
- Họ bài liên quan: *Longest Palindromic Substring* (5), *Palindrome Partitioning* (131), *Longest Palindromic Subsequence* (516), *Shortest Palindrome* (214).

## 15. Biến thể

### 15.1. Chuỗi con palindrome **dài nhất** (LeetCode 5)

Dùng cùng khung mở rộng quanh tâm, nhưng thay vì đếm, ghi nhớ đoạn dài nhất.

### 15.2. Số chuỗi con palindrome **khác nhau**

Gom các chuỗi vào `set` (hoặc cây palindrome — *eertree* — cho `O(n)`). Với `n <= 1000` dùng `set` là đủ.

### 15.3. Liệt kê toàn bộ palindrome con (tọa độ)

Mỗi lần `while` thành công thì ghi `(i, j)` thay vì cộng 1.

In [31]:
def longest_palindrome(s):
    """15.1: trả về chuỗi con palindrome dài nhất (đầu tiên nếu có nhiều)."""
    n = len(s)
    best_i, best_len = 0, 1
    for k in range(2 * n - 1):
        i, j = k // 2, (k + 1) // 2
        while i >= 0 and j < n and s[i] == s[j]:
            if j - i + 1 > best_len:
                best_i, best_len = i, j - i + 1
            i, j = i - 1, j + 1
    return s[best_i:best_i + best_len]

def count_distinct_palindromes(s):
    """15.2: số chuỗi con palindrome khác nhau."""
    n = len(s)
    seen = set()
    for k in range(2 * n - 1):
        i, j = k // 2, (k + 1) // 2
        while i >= 0 and j < n and s[i] == s[j]:
            seen.add(s[i:j + 1])
            i, j = i - 1, j + 1
    return len(seen)

def all_palindromes(s):
    """15.3: danh sách (i, j) của mọi chuỗi con palindrome."""
    n = len(s)
    out = []
    for k in range(2 * n - 1):
        i, j = k // 2, (k + 1) // 2
        while i >= 0 and j < n and s[i] == s[j]:
            out.append((i, j))
            i, j = i - 1, j + 1
    return sorted(out)

print(longest_palindrome("babad"), longest_palindrome("cbbd"), longest_palindrome("a"))
print(count_distinct_palindromes("aaa"), count_distinct_palindromes("abab"))
print(all_palindromes("aba"))

rng = random.Random(15)
for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 14)))
    ref = list_palindromes(s)
    assert all_palindromes(s) == [(i, j) for i, j, _ in ref]
    assert count_distinct_palindromes(s) == len({t for _, _, t in ref})
    assert len(longest_palindrome(s)) == max(len(t) for _, _, t in ref)
print("các biến thể 15.1-15.3 OK")

bab bb a
3 4
[(0, 0), (0, 2), (1, 1), (2, 2)]
các biến thể 15.1-15.3 OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Cắt mọi chuỗi con, so với bản đảo | `O(n^3)` | `O(n)` | Chỉ để hiểu đề / làm oracle |
| 2. DP `dp[i][j]` | `s[i]==s[j]` và trong là palindrome | `O(n^2)` | `O(n^2)` | Khi cần hỏi nhiều lần "`s[i..j]` có là palindrome không" |
| 3. Mở rộng quanh tâm | Mỗi tâm lan ra hai bên | `O(n^2)` | `O(1)` | **Nộp bài**, đơn giản, `n <= 1000` |
| 4. Manacher | Tái dùng đối xứng của palindrome lớn | `O(n)` | `O(n)` | `n` rất lớn |

In [32]:
rng = random.Random(16)
s = "".join(rng.choice("ab") for _ in range(1000))
answers = {name: fn(s) for name, fn in ALL_METHODS.items() if name != "brute"}
print(answers)
assert len(set(answers.values())) == 1

{'dp': 3098, 'expand': 3098, 'manacher': 3098, 'leetcode': 3098}


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print([(k // 2, (k + 1) // 2) for k in range(5)])
print("#" + "#".join("ab") + "#")
print(count_expand("abab"))
```

In [33]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `s = "abba"`, hãy liệt kê **tất cả** chuỗi con palindrome (kèm cặp `(i, j)`) và cho biết tổng số. Mỗi tâm `k = 0..6` đóng góp bao nhiêu?

In [34]:
# Bài 2: điền kết quả tính tay
hand_total = None                 # tổng số palindrome
hand_per_center = None            # list 7 phần tử: đóng góp của tâm k = 0..6

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm mở rộng quanh tâm.

In [35]:
def exercise_3_count(s: str) -> int:
    n = len(s)
    ans = 0
    # TODO 1: lặp k từ 0 đến 2n - 2
    #     TODO 2: i = k // 2, j = (k + 1) // 2
    #     TODO 3: while: kiểm tra BIÊN trước (i >= 0, j < n) rồi mới so s[i] == s[j]
    #         TODO 4: ans += 1, rồi i -= 1, j += 1
    return ans


# assert exercise_3_count("abc") == 3
# assert exercise_3_count("aaa") == 6
# assert exercise_3_count("abba") == 6

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 2, hãy tự viết `exercise_4_dp(s)` bằng bảng `dp[i][j]` (nhớ thứ tự duyệt `i`).

In [36]:
def exercise_4_dp(s: str) -> int:
    # TODO
    pass


# assert exercise_4_dp("abc") == 3
# assert exercise_4_dp("aaa") == 6
# assert exercise_4_dp("abcba") == 7

### Bài 5 — Mini challenge: chuỗi con palindrome dài nhất (khó)

Viết `exercise_5_longest(s)` trả về chuỗi con palindrome **dài nhất**; nếu có nhiều chuỗi cùng độ dài thì trả về chuỗi **xuất hiện sớm nhất** (chỉ số đầu nhỏ nhất). Gợi ý: dùng khung mở rộng quanh tâm.

```python
exercise_5_longest("babad") -> "bab"
exercise_5_longest("cbbd")  -> "bb"
exercise_5_longest("abc")   -> "a"
```

In [37]:
def exercise_5_longest(s: str) -> str:
    # TODO
    pass


# assert exercise_5_longest("babad") == "bab"
# assert exercise_5_longest("cbbd") == "bb"
# assert exercise_5_longest("abc") == "a"

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `k // 2` và `(k + 1) // 2` khác nhau khi `k` lẻ.
- Mức 2: `"#" + "#".join("ab") + "#"` có độ dài `2n + 1 = 5`.
- Mức 3: dòng 1 `[(0, 0), (0, 1), (1, 1), (1, 2), (2, 2)]`; dòng 2 `#a#b#`; dòng 3: `a, b, a, b, aba, bab` → `6`.

**Bài 2**
- Mức 1: bốn ký tự đơn luôn là palindrome.
- Mức 2: tâm `k = 3` (giữa hai `b`) cho `"bb"` rồi `"abba"`.
- Mức 3: tổng `6`: `a(0,0), b(1,1), b(2,2), a(3,3), bb(1,2), abba(0,3)`. Theo tâm: `[1, 0, 1, 2, 1, 0, 1]`.

**Bài 3**
- Mức 1: giống Cách 3 (mục 7.2).
- Mức 2: `for k in range(2 * n - 1)`.
- Mức 3: `while i >= 0 and j < n and s[i] == s[j]: ans += 1; i -= 1; j += 1`.

**Bài 4**
- Mức 1: `dp = [[False] * n for _ in range(n)]`.
- Mức 2: `for i in range(n - 1, -1, -1): for j in range(i, n):`.
- Mức 3: `dp[i][j] = s[i] == s[j] and (j - i < 3 or dp[i + 1][j - 1])`, đếm các ô `True`.

**Bài 5**
- Mức 1: giống đếm, nhưng thay `ans += 1` bằng cập nhật đoạn dài nhất.
- Mức 2: cập nhật khi `j - i + 1` **lớn hơn hẳn** độ dài tốt nhất hiện tại (dùng `>`, không `>=`) để giữ chuỗi sớm nhất.
- Mức 3: duyệt tâm `k` tăng dần cũng cho thứ tự xuất hiện tăng dần của chỉ số đầu với cùng độ dài; cuối cùng trả `s[best_i:best_i + best_len]`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [38]:
print([(k // 2, (k + 1) // 2) for k in range(5)])
print("#" + "#".join("ab") + "#")
print(count_expand("abab"))

assert [(k // 2, (k + 1) // 2) for k in range(5)] == [(0, 0), (0, 1), (1, 1), (1, 2), (2, 2)]
assert "#" + "#".join("ab") + "#" == "#a#b#"
assert count_expand("abab") == 6

[(0, 0), (0, 1), (1, 1), (1, 2), (2, 2)]
#a#b#
6


### Đáp án Bài 2

In [39]:
s = "abba"
ps = list_palindromes(s)
print(ps)
per_center = [0] * 7
for i, j, t in ps:
    per_center[i + j] += 1
print("theo tâm k = 0..6:", per_center)
assert len(ps) == 6
assert per_center == [1, 0, 1, 2, 1, 0, 1]

[(0, 0, 'a'), (0, 3, 'abba'), (1, 1, 'b'), (1, 2, 'bb'), (2, 2, 'b'), (3, 3, 'a')]
theo tâm k = 0..6: [1, 0, 1, 2, 1, 0, 1]


### Đáp án Bài 3

In [40]:
def exercise_3_solution(s: str) -> int:
    n = len(s)
    ans = 0
    for k in range(2 * n - 1):
        i, j = k // 2, (k + 1) // 2
        while i >= 0 and j < n and s[i] == s[j]:
            ans += 1
            i, j = i - 1, j + 1
    return ans

assert exercise_3_solution("abc") == 3
assert exercise_3_solution("aaa") == 6
assert exercise_3_solution("abba") == 6
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_solution(s: str) -> int:
    n = len(s)
    dp = [[False] * n for _ in range(n)]
    ans = 0
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            dp[i][j] = s[i] == s[j] and (j - i < 3 or dp[i + 1][j - 1])
            ans += dp[i][j]
    return ans

assert exercise_4_solution("abc") == 3
assert exercise_4_solution("aaa") == 6
assert exercise_4_solution("abcba") == 7
rng = random.Random(44)
for _ in range(200):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 12)))
    assert exercise_4_solution(s) == count_brute(s)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_solution(s: str) -> str:
    n = len(s)
    best_i, best_len = 0, 1
    for k in range(2 * n - 1):
        i, j = k // 2, (k + 1) // 2
        while i >= 0 and j < n and s[i] == s[j]:
            if j - i + 1 > best_len:
                best_i, best_len = i, j - i + 1
            i, j = i - 1, j + 1
    return s[best_i:best_i + best_len]

assert exercise_5_solution("babad") == "bab"
assert exercise_5_solution("cbbd") == "bb"
assert exercise_5_solution("abc") == "a"

def longest_oracle(s):
    best = s[0]
    for i in range(len(s)):
        for j in range(i, len(s)):
            t = s[i:j + 1]
            if t == t[::-1] and len(t) > len(best):
                best = t
    return best

rng = random.Random(55)
for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 14)))
    assert exercise_5_solution(s) == longest_oracle(s), s
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template mở rộng quanh tâm

```python
class Solution:
    def countSubstrings(self, s: str) -> int:
        n = len(s)
        ans = 0
        for k in range(2 * n - 1):            # 2n - 1 tâm: n lẻ + (n - 1) chẵn
            i, j = k // 2, (k + 1) // 2       # k chẵn: i == j; k lẻ: j == i + 1
            while i >= 0 and j < n and s[i] == s[j]:    # biên TRƯỚC, so ký tự SAU
                ans += 1                      # mỗi bước thành công = một palindrome
                i, j = i - 1, j + 1
        return ans
```

Câu thần chú: **"2n - 1 tâm, mỗi tâm lan ra tới khi gặp khác nhau; mỗi bước lan thành công là một palindrome."**

### Checklist trước khi submit

- [ ] Có **đủ hai loại tâm** (lẻ và chẵn): `2n - 1` tâm.
- [ ] Điều kiện `while` kiểm tra biên **trước** `s[i] == s[j]` (nhớ chỉ số âm không báo lỗi).
- [ ] Đếm theo **vị trí**, không loại trùng nội dung.
- [ ] Với DP: duyệt `i` giảm dần (hoặc theo độ dài tăng dần) và xử lý riêng độ dài 1, 2 (`j - i < 3`).
- [ ] Test `n = 1`, `"aa"`, `"abba"`, `"aaa..a"` (`n(n+1)/2`), chuỗi toàn khác nhau (`n`).
- [ ] Skeleton đúng: `class Solution`, method `countSubstrings(self, s: str) -> int`.